# Phase 2: Custom Feature Extraction Pipeline

**Objective:** Build a custom web scraper that can extract the exact features required by our models from any live URL. This pipeline parses the URL string, fetches the HTML content, and queries WHOIS data to construct a comprehensive feature vector.

**Outputs:**
- Custom Python functions to extract URL, HTML, and WHOIS features.
- An end-to-end `extract_features(url)` function to process live URLs.

> **Note on Dataset Alignment:**
> There are some features present in the original dataset that we do not extract in our custom pipeline:
> - `HasObfuscation`, `NoOfObfuscatedChar`, `ObfuscationRatio`, `CharContinuationRate`, `URLCharProb`
> - `NoOfSelfRedirect` is currently hardcoded to 0.
> 
> These gaps are acknowledged. Our custom scraper focuses on the most robust behavioral and structural features.

---
## 2.1 Setup & Imports

Import necessary libraries for web requests, HTML parsing (BeautifulSoup), URL parsing, and WHOIS lookups.

In [1]:
import pandas as pd
import numpy as np
import re
from urllib.parse import urlparse
import requests
from bs4 import BeautifulSoup
import whois
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

---
## 2.2 URL String Features

Extract features based purely on the structure and content of the URL string itself (length, domain, TLD, character ratios, special characters, etc.).

In [3]:
def get_url_string_features(url):
    parsed = urlparse(url)
    domain = parsed.netloc
    
    features = {}
    features['URL'] = url
    features['URLLength'] = len(url)
    features['Domain'] = domain
    features['DomainLength'] = len(domain)
    features['IsDomainIP'] = 1 if re.match(r'^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$', domain) else 0
    
    tld = domain.split('.')[-1] if '.' in domain else ''
    features['TLD'] = tld
    features['TLDLength'] = len(tld)
    features['NoOfSubDomain'] = max(0, domain.count('.') - 1)
    
    letters = sum(c.isalpha() for c in url)
    digits = sum(c.isdigit() for c in url)
    features['NoOfLettersInURL'] = letters
    features['LetterRatioInURL'] = letters / len(url) if len(url) > 0 else 0
    features['NoOfDegitsInURL'] = digits
    features['DegitRatioInURL'] = digits / len(url) if len(url) > 0 else 0
    
    features['NoOfEqualsInURL'] = url.count('=')
    features['NoOfQMarkInURL'] = url.count('?')
    features['NoOfAmpersandInURL'] = url.count('&')
    
    special_chars = len(re.findall(r'[^a-zA-Z0-9./:?&=]', url))
    features['SpacialCharRatioInURL'] = special_chars / len(url) if len(url) > 0 else 0
    
    features['IsHTTPS'] = 1 if parsed.scheme == 'https' else 0
    
    return features

# test it
test_features = get_url_string_features("https://www.google.com/search?q=test")
test_features

{'URL': 'https://www.google.com/search?q=test',
 'URLLength': 36,
 'Domain': 'www.google.com',
 'DomainLength': 14,
 'IsDomainIP': 0,
 'TLD': 'com',
 'TLDLength': 3,
 'NoOfSubDomain': 1,
 'NoOfLettersInURL': 28,
 'LetterRatioInURL': 0.7777777777777778,
 'NoOfDegitsInURL': 0,
 'DegitRatioInURL': 0.0,
 'NoOfEqualsInURL': 1,
 'NoOfQMarkInURL': 1,
 'NoOfAmpersandInURL': 0,
 'SpacialCharRatioInURL': 0.0,
 'IsHTTPS': 1}

---
## 2.3 HTML & Content Features

Fetch the webpage and extract features from the HTML content using BeautifulSoup (lines of code, external resources, forms, scripts, etc.).

In [4]:
def fetch_page(url, timeout=5):
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
    try:
        response = requests.get(url, headers=headers, timeout=timeout, allow_redirects=True)
        return response, len(response.history)  # response.history = redirect chain
    except requests.exceptions.RequestException as e:
        print(f"Could not fetch {url}: {e}")
        return None, 0

# test
resp, redirects = fetch_page("https://www.google.com")
print(resp.status_code if resp else "Failed", "| Redirects:", redirects)

200 | Redirects: 0


In [5]:
def get_html_features(response, domain):
    features = {}
    
    if response is None:
        return {
            'LineOfCode': 0, 'LargestLineLength': 0, 'HasTitle': 0, 'Title': '',
            'DomainTitleMatchScore': 0, 'HasFavicon': 0, 'NoOfPopup': 0,
            'NoOfiFrame': 0, 'HasExternalFormSubmit': 0, 'HasSocialNet': 0,
            'HasSubmitButton': 0, 'HasHiddenFields': 0, 'HasPasswordField': 0,
            'Bank': 0, 'Pay': 0, 'Crypto': 0, 'HasCopyrightInfo': 0,
            'NoOfImage': 0, 'NoOfCSS': 0, 'NoOfJS': 0,
            'NoOfSelfRef': 0, 'NoOfEmptyRef': 0, 'NoOfExternalRef': 0
        }
    
    html = response.text
    soup = BeautifulSoup(html, 'html.parser')
    lines = html.split('\n')
    
    features['LineOfCode'] = len(lines)
    features['LargestLineLength'] = max((len(l) for l in lines), default=0)
    
    title_tag = soup.find('title')
    title = title_tag.text.strip() if title_tag else ''
    features['HasTitle'] = 1 if title else 0
    features['Title'] = title
    
    # extract core domain name (strip www + TLD) and check against title
    domain_core = domain.replace('www.', '').split('.')[0]
    features['DomainTitleMatchScore'] = 100 if domain_core.lower() in title.lower() else 0
    
    features['HasFavicon'] = 1 if soup.find('link', rel=lambda x: x and 'icon' in x.lower()) else 0
    features['NoOfiFrame'] = len(soup.find_all('iframe'))
    features['NoOfPopup'] = html.lower().count('window.open')
    
    forms = soup.find_all('form')
    features['HasExternalFormSubmit'] = 1 if any(
        f.get('action', '').startswith('http') and domain not in f.get('action', '') for f in forms
    ) else 0
    features['HasSubmitButton'] = 1 if soup.find('input', type='submit') or soup.find('button', type='submit') else 0
    features['HasHiddenFields'] = 1 if soup.find('input', type='hidden') else 0
    features['HasPasswordField'] = 1 if soup.find('input', type='password') else 0
    
    social_sites = ['facebook.com', 'twitter.com', 'instagram.com', 'linkedin.com']
    features['HasSocialNet'] = 1 if any(s in html.lower() for s in social_sites) else 0
    
    features['Bank'] = 1 if 'bank' in html.lower() else 0
    features['Pay'] = 1 if 'pay' in html.lower() else 0
    features['Crypto'] = 1 if 'crypto' in html.lower() else 0
    features['HasCopyrightInfo'] = 1 if '©' in html or 'copyright' in html.lower() else 0
    
    features['NoOfImage'] = len(soup.find_all('img'))
    features['NoOfCSS'] = len(soup.find_all('link', rel='stylesheet'))
    features['NoOfJS'] = len(soup.find_all('script'))
    
    all_links = soup.find_all('a', href=True)
    self_ref, empty_ref, ext_ref = 0, 0, 0
    for link in all_links:
        href = link['href']
        if href.strip() in ('', '#'):
            empty_ref += 1
        elif domain in href or href.startswith('/'):
            self_ref += 1
        else:
            ext_ref += 1
    features['NoOfSelfRef'] = self_ref
    features['NoOfEmptyRef'] = empty_ref
    features['NoOfExternalRef'] = ext_ref
    
    return features

---
## 2.4 WHOIS Features

Extract domain registration information using WHOIS lookups to calculate features like Domain Age.

In [9]:
def get_whois_features(domain):
    features = {}
    try:
        clean_domain = domain.replace('www.', '')
        w = whois.whois(clean_domain)
        
        creation_date = w.creation_date
        if isinstance(creation_date, list):
            creation_date = creation_date[0]
        
        if creation_date:
            # strip timezone info if present, so subtraction works either way
            if creation_date.tzinfo is not None:
                creation_date = creation_date.replace(tzinfo=None)
            
            age_days = (datetime.now() - creation_date).days
            features['DomainAge'] = age_days
        else:
            features['DomainAge'] = -1  # unknown
            
    except Exception as e:
        print(f"WHOIS lookup failed for {domain}: {e}")
        features['DomainAge'] = -1
    
    return features

# test
whois_feats = get_whois_features("wikipedia.org")
whois_feats

{'DomainAge': 9330}

---
## 2.5 End-to-End Extraction Pipeline

Combine all extraction modules into a single function `extract_features(url)` that takes a URL and returns a full feature dictionary.

In [11]:
def extract_features(url):
    parsed = urlparse(url)
    domain = parsed.netloc
    
    # URL-string features
    url_feats = get_url_string_features(url)
     
    # fetch page + HTML features
    response, redirects = fetch_page(url)
    html_feats = get_html_features(response, domain)
    
    # WHOIS features
    whois_feats = get_whois_features(domain)
    
    # combine everything
    all_features = {**url_feats, **html_feats, **whois_feats}
    all_features['NoOfURLRedirect'] = redirects
    all_features['NoOfSelfRedirect'] = 0  # placeholder — refine if needed later
    
    return all_features

# full end-to-end test
result = extract_features("https://www.wikipedia.org")
for k, v in result.items():
    print(f"{k}: {v}")

URL: https://www.wikipedia.org
URLLength: 25
Domain: www.wikipedia.org
DomainLength: 17
IsDomainIP: 0
TLD: org
TLDLength: 3
NoOfSubDomain: 1
NoOfLettersInURL: 20
LetterRatioInURL: 0.8
NoOfDegitsInURL: 0
DegitRatioInURL: 0.0
NoOfEqualsInURL: 0
NoOfQMarkInURL: 0
NoOfAmpersandInURL: 0
SpacialCharRatioInURL: 0.0
IsHTTPS: 1
LineOfCode: 952
LargestLineLength: 58452
HasTitle: 1
Title: Wikipedia
DomainTitleMatchScore: 100
HasFavicon: 1
NoOfiFrame: 0
NoOfPopup: 0
HasExternalFormSubmit: 0
HasSubmitButton: 1
HasHiddenFields: 1
HasPasswordField: 0
HasSocialNet: 0
Bank: 0
Pay: 0
Crypto: 0
HasCopyrightInfo: 1
NoOfImage: 1
NoOfCSS: 0
NoOfJS: 4
NoOfSelfRef: 366
NoOfEmptyRef: 2
NoOfExternalRef: 8
DomainAge: 9330
NoOfURLRedirect: 0
NoOfSelfRedirect: 0


---
## Summary

### What was accomplished:
- ✅ Built Python functions to parse raw URLs (`get_url_string_features`)
- ✅ Implemented secure web request handling and HTML parsing (`get_html_features`)
- ✅ Added WHOIS integration for domain metadata (`get_whois_features`)
- ✅ Assembled an end-to-end `extract_features(url)` pipeline ready to ingest live data for model inference.

### Next step: Phase 3 — Preprocessing & Feature Engineering (using the main dataset)